# Notebook 22 — Visualización comparativa: frame original vs SAM2

**TFM — Sistema de Detección de Amenazas Armadas en Vídeo**  
Oliver Legarreta García · Universitat Oberta de Catalunya

---

Genera vídeos lado a lado mostrando qué ve cada pipeline en cada frame:
- **Izquierda:** frame original con bbox de persona y detección de arma (Config A)
- **Derecha:** frame con overlay de máscaras SAM2 y detección de arma sobre cada segmento

**Clips seleccionados:**

| Clip | Categoría | Config A | Config SAM2 | Interés |
|------|----------|----------|-------------|--------|
| `N10_C2_P5_V4_HB_2` | Botella de agua | TN | **FP** ❌ | SAM2 confunde botella con arma |
| `PAH7_C2_P5_V1_HB_2` | Pistola apuntando | TP | **TP** ✅ | SAM2 detecta correctamente |

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install sam2 ultralytics opencv-python
print('✅ Dependencias instaladas')

In [ ]:
import os
import shutil
import numpy as np
import cv2
import torch
from pathlib import Path
from ultralytics import YOLO
from sam2.sam2_image_predictor import SAM2ImagePredictor
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ── CONFIG ────────────────────────────────────────────────────────────────────
BASE_DIR       = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset'
WEAPON_WEIGHTS = '/content/drive/MyDrive/TFM/experiments/weapon_det/yolov8m_weapons_B_e50_640/weights/best.pt'
OUT_DIR        = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/visualizacion'

SAM2_MODEL_ID       = 'facebook/sam2.1-hiera-base-plus'
CONF_SEG            = 0.25
CONF_WEAPON         = 0.25
IOU_NMS             = 0.7
PADDING             = 0.10
MIN_MASK_AREA_RATIO = 0.02
MAX_MASKS           = 5
MAX_SECONDS         = 15

# Clips a visualizar
CLIPS = {
    'FP_SAM2': {
        'name':    'N10_C2_P5_V4_HB_2',
        'label':   'N10 — Botella de agua (TN en A, FP en SAM2)',
        'true':    0,
        'result_A':   'TN',
        'result_SAM2':'FP',
    },
    'TP_SAM2': {
        'name':    'PAH7_C2_P5_V1_HB_2',
        'label':   'PAH7 — Pistola apuntando (TP en A y SAM2)',
        'true':    1,
        'result_A':   'TP',
        'result_SAM2':'TP',
    },
}

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('✅ Config cargada')

---
## 1. Carga de modelos

In [ ]:
!cp '{WEAPON_WEIGHTS}' /content/weapon_best.pt

seg_model    = YOLO('yolov8s-seg.pt')
weapon_model = YOLO('/content/weapon_best.pt')
sam2_pred    = SAM2ImagePredictor.from_pretrained(SAM2_MODEL_ID)

print('✅ Modelos cargados')

---
## 2. Funciones de anotación visual

In [ ]:
# Paleta de colores para máscaras SAM2
MASK_COLORS = [
    (255, 100, 100), (100, 255, 100), (100, 100, 255),
    (255, 255, 100), (255, 100, 255),
]

def draw_config_a(frame, frame_idx):
    """
    Dibuja sobre el frame la visualización de Config A:
    - bbox de persona (azul)
    - detección de arma si existe (rojo)
    """
    out = frame.copy()
    h, w = out.shape[:2]

    # Detectar persona
    seg_res = seg_model.predict(frame, imgsz=640, conf=CONF_SEG,
                                classes=[0], verbose=False, device='cuda')[0]
    if seg_res.boxes is not None and len(seg_res.boxes) > 0:
        # Persona de mayor área
        best_box, best_area = None, 0
        for box in seg_res.boxes:
            x1,y1,x2,y2 = map(float, box.xyxy[0].cpu())
            area = (x2-x1)*(y2-y1)
            if area > best_area:
                best_area, best_box = area, (x1,y1,x2,y2)
        if best_box:
            x1,y1,x2,y2 = map(int, best_box)
            cv2.rectangle(out, (x1,y1), (x2,y2), (255,165,0), 2)
            cv2.putText(out, 'persona', (x1,y1-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,165,0), 1)

    # Detectar arma en frame completo
    wpn_res = weapon_model.predict(frame, imgsz=640, conf=CONF_WEAPON,
                                   iou=IOU_NMS, verbose=False, device='cuda')[0]
    weapon_found = False
    if wpn_res.boxes is not None and len(wpn_res.boxes) > 0:
        weapon_found = True
        for box in wpn_res.boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cv2.rectangle(out, (x1,y1), (x2,y2), (0,0,255), 2)
            cv2.putText(out, f'arma {conf:.2f}', (x1,y1-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)

    # Header
    status = 'ARMA ✓' if weapon_found else 'sin arma'
    color  = (0,0,255) if weapon_found else (0,200,0)
    cv2.rectangle(out, (0,0), (w,28), (0,0,0), -1)
    cv2.putText(out, f'Config A — frame {frame_idx} — {status}',
                (6,18), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1)
    return out, weapon_found


def draw_config_sam2(frame, frame_idx):
    """
    Dibuja sobre el frame la visualización de Config SAM2:
    - bbox de persona con padding (naranja)
    - máscaras SAM2 con colores distintos
    - borde rojo en máscaras clasificadas como arma
    """
    out = frame.copy()
    h, w = out.shape[:2]
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detectar persona
    seg_res = seg_model.predict(frame, imgsz=640, conf=CONF_SEG,
                                classes=[0], verbose=False, device='cuda')[0]
    bbox = None
    if seg_res.boxes is not None and len(seg_res.boxes) > 0:
        best_box, best_area = None, 0
        for box in seg_res.boxes:
            x1,y1,x2,y2 = map(float, box.xyxy[0].cpu())
            area = (x2-x1)*(y2-y1)
            if area > best_area:
                best_area, best_box = area, (x1,y1,x2,y2)
        if best_box:
            px1,py1,px2,py2 = best_box
            bw,bh = px2-px1, py2-py1
            px1 = max(0, int(px1-bw*PADDING))
            py1 = max(0, int(py1-bh*PADDING))
            px2 = min(w, int(px2+bw*PADDING))
            py2 = min(h, int(py2+bh*PADDING))
            bbox = (px1,py1,px2,py2)
            cv2.rectangle(out, (px1,py1), (px2,py2), (255,165,0), 2)
            cv2.putText(out, 'bbox+C10', (px1,py1-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,165,0), 1)

    weapon_found = False
    n_masks_drawn = 0

    if bbox is not None:
        # SAM2
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
            sam2_pred.set_image(frame_rgb)
            input_box = np.array([[bbox[0],bbox[1],bbox[2],bbox[3]]], dtype=np.float32)
            masks, scores, _ = sam2_pred.predict(
                point_coords=None, point_labels=None,
                box=input_box, multimask_output=True
            )

        bbox_area = (bbox[2]-bbox[0])*(bbox[3]-bbox[1])
        min_area  = bbox_area * MIN_MASK_AREA_RATIO

        mask_list = [(m.sum(), m.astype(np.uint8)) for m in masks if m.sum() >= min_area]
        mask_list.sort(key=lambda x: x[0], reverse=True)
        mask_list = mask_list[:MAX_MASKS]

        for idx, (area, mask) in enumerate(mask_list):
            color = MASK_COLORS[idx % len(MASK_COLORS)]

            # Overlay semitransparente
            overlay = out.copy()
            overlay[mask == 1] = (
                overlay[mask == 1] * 0.4 +
                np.array(color[::-1]) * 0.6  # BGR
            ).astype(np.uint8)
            out = overlay

            # Contorno de la máscara
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(out, contours, -1, color[::-1], 1)

            # Pasar máscara al weapon_model
            masked_frame = np.zeros_like(frame)
            masked_frame[mask == 1] = frame[mask == 1]
            wpn_res = weapon_model.predict(
                masked_frame, imgsz=640, conf=CONF_WEAPON,
                iou=IOU_NMS, verbose=False, device='cuda'
            )[0]

            if wpn_res.boxes is not None and len(wpn_res.boxes) > 0:
                weapon_found = True
                for box in wpn_res.boxes:
                    x1,y1,x2,y2 = map(int, box.xyxy[0])
                    conf = float(box.conf[0])
                    cv2.rectangle(out, (x1,y1), (x2,y2), (0,0,255), 2)
                    cv2.putText(out, f'ARMA {conf:.2f}', (x1,y1-6),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
            n_masks_drawn += 1
    else:
        # Fallback frame completo
        wpn_res = weapon_model.predict(frame, imgsz=640, conf=CONF_WEAPON,
                                       iou=IOU_NMS, verbose=False, device='cuda')[0]
        if wpn_res.boxes is not None and len(wpn_res.boxes) > 0:
            weapon_found = True
            for box in wpn_res.boxes:
                x1,y1,x2,y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cv2.rectangle(out, (x1,y1), (x2,y2), (0,0,255), 2)
                cv2.putText(out, f'ARMA {conf:.2f}', (x1,y1-6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)

    # Header
    status = 'ARMA ✓' if weapon_found else 'sin arma'
    color  = (0,0,255) if weapon_found else (0,200,0)
    cv2.rectangle(out, (0,0), (w,28), (0,0,0), -1)
    cv2.putText(out, f'SAM2 — frame {frame_idx} — {n_masks_drawn} máscaras — {status}',
                (6,18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return out, weapon_found


print('✅ Funciones de anotación definidas')

---
## 3. Función principal: generar vídeo lado a lado

In [ ]:
def find_video_path(clip_name):
    """Busca el fichero de vídeo del clip en el dataset GAR."""
    # Estructura: BASE_DIR/{categoria}/{clip_name}/{clip_name}.mp4 o similar
    cat = clip_name.split('_')[0]  # PAH7, N10, etc.
    candidates = list(Path(BASE_DIR).rglob(f'{clip_name}*.mp4'))
    if not candidates:
        candidates = list(Path(BASE_DIR).rglob(f'{clip_name}*.avi'))
    if candidates:
        return candidates[0]
    return None


def generate_side_by_side_video(clip_info, stride=3, max_seconds=MAX_SECONDS):
    """
    Genera un vídeo lado a lado:
    Izquierda: Config A | Derecha: Config SAM2
    stride=3: procesar 1 de cada 3 frames para reducir tiempo
    """
    clip_name = clip_info['name']
    label     = clip_info['label']

    # Buscar vídeo
    vp = find_video_path(clip_name)
    if vp is None:
        print(f'❌ Vídeo no encontrado: {clip_name}')
        return

    local_in = '/content/tmp_vis.mp4'
    shutil.copy2(str(vp), local_in)

    cap = cv2.VideoCapture(local_in)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    max_frames = n_frames
    if max_seconds and fps > 0:
        max_frames = min(max_frames, int(max_seconds * fps))

    # Redimensionar para que quepan dos frames lado a lado
    target_w = 480
    target_h = int(orig_h * target_w / orig_w)
    out_w = target_w * 2
    out_h = target_h + 40  # espacio para título inferior

    out_path = f'{OUT_DIR}/{clip_name}_sbs.mp4'
    writer = cv2.VideoWriter(
        out_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps / stride,
        (out_w, out_h)
    )

    frame_i = 0
    det_a, det_sam2 = 0, 0

    print(f'  Procesando {clip_name} ({max_frames} frames, stride={stride})...')

    while True:
        ok, frame = cap.read()
        if not ok or frame_i >= max_frames:
            break

        if frame_i % stride == 0:
            # Anotar ambas configs
            frame_a,    gun_a    = draw_config_a(frame, frame_i)
            frame_sam2, gun_sam2 = draw_config_sam2(frame, frame_i)

            if gun_a:    det_a    += 1
            if gun_sam2: det_sam2 += 1

            # Redimensionar
            frame_a    = cv2.resize(frame_a,    (target_w, target_h))
            frame_sam2 = cv2.resize(frame_sam2, (target_w, target_h))

            # Combinar lado a lado
            combined = np.zeros((out_h, out_w, 3), dtype=np.uint8)
            combined[:target_h, :target_w]          = frame_a
            combined[:target_h, target_w:out_w]     = frame_sam2

            # Línea separadora
            combined[:target_h, target_w-1:target_w+1] = (200,200,200)

            # Barra inferior con resultado acumulado
            cv2.rectangle(combined, (0, target_h), (out_w, out_h), (30,30,30), -1)
            res_a    = f'A: {det_a} frames'
            res_sam2 = f'SAM2: {det_sam2} frames'
            cv2.putText(combined, res_a,    (10, target_h+26),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)
            cv2.putText(combined, res_sam2, (target_w+10, target_h+26),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)

            # Umbral de clasificación
            thr_color_a    = (0,0,255) if det_a    >= 5 else (0,200,0)
            thr_color_sam2 = (0,0,255) if det_sam2 >= 5 else (0,200,0)
            cv2.putText(combined, f'→ {clip_info["result_A"]}',
                        (200, target_h+26), cv2.FONT_HERSHEY_SIMPLEX,
                        0.55, thr_color_a, 1)
            cv2.putText(combined, f'→ {clip_info["result_SAM2"]}',
                        (target_w+200, target_h+26), cv2.FONT_HERSHEY_SIMPLEX,
                        0.55, thr_color_sam2, 1)

            writer.write(combined)

        frame_i += 1

    cap.release()
    writer.release()
    os.remove(local_in)

    print(f'  ✅ Guardado en: {out_path}')
    print(f'     Config A:    {det_a} frames con arma → {clip_info["result_A"]}')
    print(f'     Config SAM2: {det_sam2} frames con arma → {clip_info["result_SAM2"]}')
    return out_path


print('✅ Función de vídeo definida')

---
## 4. Generar vídeo 1 — N10 Botella (TN en A, FP en SAM2)

In [ ]:
print('=' * 55)
print('VÍDEO 1 — Botella de agua: SAM2 comete un FP')
print('=' * 55)
print(CLIPS['FP_SAM2']['label'])
print()
path_fp = generate_side_by_side_video(CLIPS['FP_SAM2'], stride=3)

---
## 5. Generar vídeo 2 — PAH7 Pistola apuntando (TP en A y SAM2)

In [ ]:
print('=' * 55)
print('VÍDEO 2 — Pistola apuntando: ambos configs aciertan')
print('=' * 55)
print(CLIPS['TP_SAM2']['label'])
print()
path_tp = generate_side_by_side_video(CLIPS['TP_SAM2'], stride=3)

---
## 6. Frames clave — comparativa estática

Extrae los frames más representativos de cada vídeo para incluir en la memoria o presentación.

In [ ]:
def extract_key_frames(clip_info, n_frames=6):
    """
    Extrae n_frames representativos del clip y los muestra en un grid.
    Selecciona frames donde SAM2 tiene detección (para el caso FP) o
    donde hay más máscaras activas.
    """
    clip_name = clip_info['name']
    vp = find_video_path(clip_name)
    if vp is None:
        print(f'❌ Vídeo no encontrado: {clip_name}')
        return

    local_in = '/content/tmp_kf.mp4'
    shutil.copy2(str(vp), local_in)

    cap = cv2.VideoCapture(local_in)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_frames = min(n_total, int(MAX_SECONDS * fps) if fps > 0 else n_total)

    # Seleccionar índices equiespaciados
    indices = np.linspace(10, max_frames - 10, n_frames, dtype=int)

    frames_a    = []
    frames_sam2 = []
    frame_i = 0
    idx_set = set(indices)

    while True:
        ok, frame = cap.read()
        if not ok or frame_i >= max_frames:
            break
        if frame_i in idx_set:
            fa, _    = draw_config_a(frame, frame_i)
            fs, _    = draw_config_sam2(frame, frame_i)
            frames_a.append(cv2.cvtColor(fa, cv2.COLOR_BGR2RGB))
            frames_sam2.append(cv2.cvtColor(fs, cv2.COLOR_BGR2RGB))
        frame_i += 1

    cap.release()
    os.remove(local_in)

    # Grid: 2 filas (A arriba, SAM2 abajo) × n_frames columnas
    fig, axes = plt.subplots(2, n_frames, figsize=(n_frames * 3.5, 5))
    for j in range(len(frames_a)):
        axes[0, j].imshow(frames_a[j])
        axes[0, j].axis('off')
        axes[0, j].set_title(f'f={indices[j]}', fontsize=8)
        axes[1, j].imshow(frames_sam2[j])
        axes[1, j].axis('off')

    axes[0, 0].set_ylabel('Config A', fontsize=10, rotation=90, labelpad=4)
    axes[1, 0].set_ylabel('SAM2', fontsize=10, rotation=90, labelpad=4)

    plt.suptitle(f'{clip_name} — {clip_info["label"]}\n'
                 f'A={clip_info["result_A"]} | SAM2={clip_info["result_SAM2"]}',
                 fontsize=11)
    plt.tight_layout()
    save_path = f'{OUT_DIR}/{clip_name}_keyframes.png'
    plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'✅ Guardado: {save_path}')


print('Extrayendo frames clave — Botella (FP SAM2)...')
extract_key_frames(CLIPS['FP_SAM2'], n_frames=6)

In [ ]:
print('Extrayendo frames clave — Pistola (TP SAM2)...')
extract_key_frames(CLIPS['TP_SAM2'], n_frames=6)

---

## Interpretación visual

**Vídeo 1 — Botella (FP en SAM2):**  
En el lado izquierdo (Config A) se puede ver que el detector de armas apenas activa sobre la botella en el frame completo. En el lado derecho (SAM2) las máscaras de color aíslan la botella como objeto individual — y el weapon_model, al ver la botella sin contexto, la clasifica como arma. El FP viene del aislamiento excesivo: sin la mano, el cuerpo y el entorno alrededor, la botella tiene una silueta que el modelo confunde con un arma.

**Vídeo 2 — Pistola apuntando (TP en ambos):**  
En el lado izquierdo el detector localiza el arma en el frame completo. En el lado derecho SAM2 aísla el arma como segmento y el weapon_model la detecta correctamente. En este caso el aislamiento ayuda — el arma extendida ocupa un segmento claro y diferenciado.